# 03. 추적 영상 만들기 (탐지 + 추적 + 궤적 + 정지비행)

학습된 `models/best.pt`로 테스트 영상을 추적하고, 과제 제출용 결과 영상과 로그를 만듭니다.

**산출물**
- `videos/beeguard_tracking.mp4` — 박스·추적 ID·궤적·말벌 정지비행(HOVER) 표시 (H.264, GitHub에서 재생 가능)
- `results/demo.gif` — README 맨 위에 넣을 GIF
- `logs/tracks.csv` — 프레임별 좌표와 속도 (교수님 참고자료의 "trajectory 따로 저장")
- `logs/detections.csv` — 대시보드 '최근 탐지 기록' 형식 이벤트 로그
- `results/detection_timeline.png` — 시간에 따른 부류별 탐지 수와 정지비행 구간

**주의**: 테스트 영상의 프레임은 학습 데이터에 들어가면 안 됩니다.

In [ ]:
# ===== 셀 1: 환경 준비 =====
REPO_URL = "https://github.com/djkdb/beeguard-hornet-tracker.git"   # 본인 저장소 주소로 바꾸기
!pip -q install ultralytics lap
import os, subprocess
if not os.path.isdir("/content/repo"):
    subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=False)
REPO = "/content/repo" if os.path.isdir("/content/repo") else "/content"
os.chdir(REPO)
for d in ["models", "videos/clips", "logs", "results"]:
    os.makedirs(d, exist_ok=True)
print("REPO:", REPO, "| best.pt 있음:", os.path.exists("models/best.pt"))

In [ ]:
# ===== 셀 2: best.pt 가져오기 (02_train.ipynb 셀 10이 Drive에 저장한 것, 없으면 업로드) =====
import shutil
DRIVE_DIR = "/content/drive/MyDrive/beeguard"
if not os.path.exists("models/best.pt"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        for f, dst in [("best.pt", "models/best.pt"), ("experiments.csv", "results/experiments.csv")]:
            if os.path.exists(f"{DRIVE_DIR}/{f}"):
                shutil.copy(f"{DRIVE_DIR}/{f}", dst)
    except ImportError:
        pass
if not os.path.exists("models/best.pt"):
    from google.colab import files
    up = files.upload()
    os.replace(next(iter(up)), "models/best.pt")
print("best.pt:", os.path.getsize("models/best.pt") // 1024, "KB")

In [ ]:
# ===== 셀 3: 영상 클립 준비 → 잘라서 이어 붙이기 =====
# 기본값: 라이선스가 명확한 공개 클립을 자동으로 내려받습니다 (README '출처'에 기록됨).
# 직접 찍은 영상이나 다른 클립을 쓰려면 USE_UPLOAD = True로 바꾸고 CLIPS를 고치세요.
# ※ 테스트 영상의 프레임은 학습 데이터에 들어가면 안 됩니다.
USE_UPLOAD = False
UA = "BeeGuardStudentProject/0.2 (https://github.com/djkdb/beeguard-hornet-tracker)"
CLIP_SOURCES = {   # 파일 이름: (내려받을 주소, 출처 페이지, 작성자, 라이선스)
    "bees_mixkit_25173.mp4": (
        "https://assets.mixkit.co/videos/25173/25173-720.mp4",
        "https://mixkit.co/free-stock-video/bees-at-the-entrance-of-the-beehive-25173/", "azgek", "Mixkit Stock Video Free License"),
    "giant_hornet_naturebox.webm": (
        "https://upload.wikimedia.org/wikipedia/commons/transcoded/a/ab/Japanese_giant_hornet.webm/Japanese_giant_hornet.webm.480p.vp9.webm",
        "https://commons.wikimedia.org/wiki/File:Japanese_giant_hornet.webm", "The Nature Box", "CC BY-SA 3.0"),
    "yellow_legged_hornet_ddfrelons.webm": (
        "https://upload.wikimedia.org/wikipedia/commons/transcoded/8/8d/Un_frelon_COINCE_son_DARD_dans_ma_GoPro.webm/Un_frelon_COINCE_son_DARD_dans_ma_GoPro.webm.720p.vp9.webm",
        "https://commons.wikimedia.org/wiki/File:Un_frelon_COINCE_son_DARD_dans_ma_GoPro.webm", "DD Frelons", "CC BY 3.0"),
}
# (파일 이름, 시작 초, 길이 초) — 순서대로 이어 붙임
CLIPS = [
    ("bees_mixkit_25173.mp4", 0, 5),          # 벌통 입구의 꿀벌 무리
    ("giant_hornet_naturebox.webm", 86, 5),   # 1:26~1:31 나뭇가지 위 장수말벌
    ("yellow_legged_hornet_ddfrelons.webm", 160, 5),   # 2:40~2:45 지붕 아래 둥지 위의 등검은말벌 여러 마리
]

if USE_UPLOAD:
    from google.colab import files
    up = files.upload()
    for name in up:
        os.replace(name, os.path.join("videos/clips", name))
else:
    for name, (url, page, author, lic) in CLIP_SOURCES.items():
        path = os.path.join("videos/clips", name)
        if not os.path.exists(path):
            subprocess.run(["curl", "-fsSL", "--retry", "5", "--retry-delay", "20", "--retry-all-errors",
                            "-A", UA, "-o", path, url], check=True)   # 429 등 오류 응답은 재시도, HTML 오류 페이지를 저장하지 않음
        print(f"{name}: {os.path.getsize(path) // 1024} KB | {author} · {lic} | {page}")
print(os.listdir("videos/clips"))

args = []
for name, start, dur in CLIPS:
    args += ["--clip", f"videos/clips/{name}", str(start), str(dur)]
subprocess.run(["python", "scripts/video_tools.py", "cut", *args, "--out", "videos/clips/test.mp4"], check=True)

In [ ]:
# ===== 셀 4: 추적 실행 =====
import pandas as pd
IMGSZ = 960
if os.path.exists("results/experiments.csv"):                 # 02에서 고른 모델의 입력 크기와 맞춤
    exp = pd.read_csv("results/experiments.csv").sort_values("mAP50-95", ascending=False).iloc[0]
    IMGSZ = int(exp["imgsz"])
print("IMGSZ:", IMGSZ)

!python scripts/track_trajectory.py --model models/best.pt --source videos/clips/test.mp4 \
    --out videos/beeguard_tracking_raw.mp4 --logs logs --imgsz {IMGSZ} --conf 0.25 \
    --tracker bytetrack.yaml --hover-window 1.0 --hover-speed 1.0 --hover-min 1.0

In [ ]:
# ===== 셀 5: GitHub에서 재생되는 H.264 영상 + README용 GIF =====
!python scripts/video_tools.py h264 videos/beeguard_tracking_raw.mp4 videos/beeguard_tracking.mp4
!python scripts/video_tools.py gif videos/beeguard_tracking.mp4 results/demo.gif --width 640 --fps 10
os.remove("videos/beeguard_tracking_raw.mp4")

from IPython.display import Image as IPImage
IPImage(filename="results/demo.gif")

In [ ]:
# ===== 셀 6: 시간에 따른 탐지 수와 정지비행 구간 그래프 =====
import json
import matplotlib.pyplot as plt

tr = pd.read_csv("logs/tracks.csv", encoding="utf-8-sig")
tr["sec"] = tr["time_s"].astype(float).astype(int)
counts = tr.groupby(["sec", "class"])["track_id"].nunique().unstack(fill_value=0)
LABEL = {"honeybee": "honeybee", "giant_hornet": "giant hornet", "yellow_legged_hornet": "yellow-legged hornet"}
COLOR = {"honeybee": "#F2B705", "giant_hornet": "#D62828", "yellow_legged_hornet": "#6A4C93"}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, gridspec_kw={"height_ratios": [3, 2]})
for c in counts.columns:
    ax1.plot(counts.index, counts[c], marker="o", label=LABEL.get(c, c), color=COLOR.get(c))
ax1.set_ylabel("objects per second")
ax1.legend(loc="upper right")
ax1.set_title("Detections over time")

hov = tr[(tr["hovering"] == 1) & (tr["class"].isin(["giant_hornet", "yellow_legged_hornet"]))]
for i, (tid, g) in enumerate(hov.groupby("track_id")):
    ax2.scatter(g["time_s"], [i] * len(g), s=6, color=COLOR.get(g["class"].iloc[0]))
    ax2.text(g["time_s"].min(), i + 0.15, f"id {tid}", fontsize=8)
ax2.set_ylim(-0.6, max(hov["track_id"].nunique(), 1) - 0.2)
ax2.set_ylabel("hornet tracks")
ax2.set_xlabel("time (s)")
ax2.set_title("Hovering intervals (hornets)")
ax2.set_yticks([])
plt.tight_layout()
plt.savefig("results/detection_timeline.png", dpi=150)
plt.show()

print(json.dumps(json.load(open("logs/summary.json", encoding="utf-8")), ensure_ascii=False, indent=2))
pd.read_csv("logs/detections.csv", encoding="utf-8-sig").head(15)

In [ ]:
# ===== 셀 7: 결과 내려받기 (또는 git으로 커밋) =====
import zipfile
KEEP = ["models/best.pt", "videos/beeguard_tracking.mp4", "results", "logs"]
with zipfile.ZipFile("/content/beeguard_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for item in KEEP:
        if os.path.isfile(item):
            z.write(item)
        for root, _, fs in os.walk(item):
            for f in fs:
                z.write(os.path.join(root, f))
print("beeguard_outputs.zip:", os.path.getsize("/content/beeguard_outputs.zip") // 1024, "KB")
try:
    from google.colab import files
    files.download("/content/beeguard_outputs.zip")
except ImportError:
    pass

## 결과 확인 체크리스트
- [ ] 영상 길이가 5초 이상이고, 세 부류 중 최소 두 부류 이상이 등장한다
- [ ] 같은 개체의 ID가 유지되는지, 바뀌는 장면이 있다면 그 시점(`logs/tracks.csv`)을 보고서에 적는다
- [ ] `summary.json`의 `tracks_with_class_switch`가 0이 아니면, 종 판정이 흔들린 장면을 실패 사례로 분석한다
- [ ] 정지비행 기준(`--hover-speed`, `--hover-min`)을 바꿨다면 그 값과 이유를 보고서에 적는다
- [ ] README에 영상 출처(사이트, 작성자, 라이선스)를 적는다